# gene-tidy — clean messy gene/protein identifier tables

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MargoSolo/gene-tidy/blob/main/notebooks/gene_tidy_colab.ipynb)
[![PyPI](https://img.shields.io/pypi/v/gene-tidy)](https://pypi.org/project/gene-tidy/)

> **Messy gene table in → clean, audited table out. Offline. No silent guessing.**

`SEPT2` turned into `2-Sep`? `p53` isn't an official symbol? An old name like
`FRAP1` quietly breaking your joins? gene-tidy fixes the fixable, **flags** the
genuinely ambiguous, and isolates the junk — and every single decision is
traceable.

**Zero setup.** Run the cells top to bottom:

1. Install gene-tidy
2. Pick a file (a bundled messy example, or upload your own)
3. Run the cleaner
4. Preview the clean / ambiguous / failed rows
5. Download a ZIP with all six output files

Every identifier is mapped to the current **HGNC approved symbol** plus
Ensembl / UniProt / Entrez / RefSeq cross-references, Excel date-corrupted
symbols are recovered (`SEPT2 → "2-Sep"`), and the tool **never guesses silently
or drops a row** — running **fully offline** against a bundled HGNC dump.

## 1. Install

In [ ]:
# gene-tidy is on PyPI — one line, no setup:
%pip install -q gene-tidy
# (Want the bleeding edge instead? %pip install -q "git+https://github.com/MargoSolo/gene-tidy.git")

## ✨ 60-second taste — messy in, clean out

One hand-picked table that hits every case: an approved symbol, an alias, an
Ensembl ID, an outdated symbol, an Excel-mangled name, two genuinely ambiguous
corruptions, and one piece of junk. Watch what each one becomes 👇

In [ ]:
from gene_tidy import tidy_values

# A tiny hand-picked table that hits every case at once.
messy = ["TP53", "p53", "ENSG00000141510", "HER2", "FRAP1",
         "Sep-7", "2-Sep", "1-Mar", "FOOBAR1"]

taste = tidy_values(messy)
view = taste.audit[["input_value", "detected_type", "approved_symbol", "match_status"]].copy()
view.columns = ["you typed", "detected as", "→ approved symbol", "status"]

c = taste.counts
print(f"{c['total']} in  ->  {c['clean']} clean, "
      f"{c['ambiguous']} ambiguous (flagged for review), {c['failed']} failed."
      "   Nothing dropped.")
view

## 2. Choose your input

By default we use the **bundled messy example** so you can click *Run* and see
results immediately. To use your own file, run the upload cell below it.

In [ ]:
from gene_tidy.examples import EXAMPLE_PATH
import pandas as pd

input_path = str(EXAMPLE_PATH)  # the bundled messy_example.xlsx
print('Using bundled example:', input_path)
pd.read_excel(input_path).head(25)

In [ ]:
# OPTIONAL: upload your own .txt / .csv / .tsv / .xlsx instead.
# (Skip this cell to keep using the bundled example.)
try:
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    if uploaded:
        input_path = next(iter(uploaded))
        print('Using uploaded file:', input_path)
except ImportError:
    print('Not on Colab — keeping', input_path)

## 3. Run the cleaner

In [ ]:
from gene_tidy import tidy_file

result = tidy_file(input_path, 'outputs/')

print('HGNC dump:', result.hgnc_version.get('hgnc_release'),
      '(retrieved', result.hgnc_version.get('downloaded_date'), ')')
print('Identifier column(s):', ', '.join(result.id_columns))
print('Counts:', result.counts)

## 4. Preview the results

### ✅ Clean rows (confidently resolved)

In [ ]:
cols = ['input_value', 'detected_type', 'approved_symbol', 'hgnc_id',
        'ensembl_gene_id', 'uniprot_id', 'entrez_id', 'match_status', 'warning']
result.clean[cols]

### ⚠️ Ambiguous rows (need manual review — never auto-resolved)

In [ ]:
result.ambiguous[['input_value', 'detected_type', 'approved_symbol',
                  'match_status', 'warning']]

### ❌ Failed rows (no match)

In [ ]:
result.failed[['input_value', 'detected_type', 'match_status', 'warning']]

### 📝 Methods paragraph (paste into your supplementary methods)

In [ ]:
print(result.methods_text)

## 5. Download all outputs as a ZIP

In [ ]:
import shutil

zip_path = shutil.make_archive('gene_tidy_outputs', 'zip', 'outputs')
print('Wrote', zip_path)

try:
    from google.colab import files  # type: ignore
    files.download(zip_path)
except ImportError:
    print('Not on Colab — find the ZIP and the outputs/ folder in the file browser.')